# Load libraries

In [ ]:
import os
import sys
import time
import datetime

from termcolor import colored, cprint

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import sys
from os import listdir

import numpy as np
import pandas as pd  # Import Pandas and Numpy to create databases
import tensorflow as tf  # Import tensorflow to do tensor manipulations
import matplotlib.pyplot as plt
from abyss.helpers import metrics_viewer
from abyss.models import create_decoder
from abyss.glob import glob
import statsmodels.api as sm
import seaborn as sns
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [ ]:
phenotypes_labels = {
    "bc": "Breast Cancer",
    "cad": "Coronary Artery Disease",
    "ms": "Multiple Sclerosis",
    "t2d": "Type 2 Diabetes",
    "pc": "Prostate Cancer",
    "ibd": "Inflammatory Bowel Disease"
}

scenario_labels = {
    "no_cov": "no covariates",
    "abyss": "Abyss correction",
    "cov": "PCA correction",
    "abyss_cov": "Abyss + PCA as covariates"
}

In [ ]:
max_maf = 0.5
min_maf = 0
logp_min = 6
logp_max = 100
P_cutoff = 5e-8
PCs = 15
nr_snps = 10000

In [ ]:
bin = 1

In [ ]:
bin_param = f"bin_{bin}_sumstat.pkl"

In [ ]:
def filter_bins(bin_list, bin_param):
    if bin_param in bin_list:
        return [bin_param]
    else:
        return bin_list

In [ ]:
disease = "bc"
scenario = "no_cov"

In [ ]:
path_input = glob.PATH_GENO + f"/09_summary_statistics/{disease}/{scenario}"

In [ ]:
list_of_bins = os.listdir(path_input)

In [ ]:
#try:
#    bin = 1
#    bin_param = f"bin_{bin}_sumstat.pkl"
#    list_of_bins = filter_bins(list_of_bins, bin_param)
#except:
#    bin = "total"

In [ ]:
path_output = glob.PATH_PHENO

In [ ]:
path_output = path_output + f"/PRS/{disease}/bin_{bin}"
os.makedirs(path_output, exist_ok=True)

In [ ]:
list_of_dfs = []
for biny in list_of_bins:
    try:
        temp = pd.read_pickle(path_input+f"/{biny}")
        temp['bin'] = biny.split('_')[1]
        
        # Sort by -logp in descending order and take the top 1000 SNPs
        #top_snps = temp.sort_values(by='-logp', ascending=False).head(nr_snps)
        top_snps = temp[temp['-logp']>3]
        list_of_dfs.append(top_snps)
    except Exception as e:
        print(e)
        print(f"Problem with chrom {biny}")



In [ ]:
# Concatenating the top SNPs from each bin
sumstat = pd.concat(list_of_dfs)

# Sorting by chromosome if needed
sumstat = sumstat.sort_values(by='#CHROM', ascending=True)

In [ ]:
sumstat.reset_index(drop=True, inplace=True)
sumstat

In [ ]:
sumstat = sumstat[sumstat['-logp'] < 300]

In [ ]:
running_pos = 0
cumulative_pos = []
for chrom, group_df in sumstat.groupby('#CHROM'):
    cumulative_pos.append(group_df['POS'] + running_pos)
    running_pos += group_df['POS'].max()

sumstat['cumulative_pos'] = pd.concat(cumulative_pos)
sumstat['SNP number'] = sumstat.index

# Define two contrasting colors from the rocket palette
palette = sns.color_palette("rocket", as_cmap=False)
color1 = palette[3]  # First color of the palette
color2 = palette[0] # Last color of the palette, for maximum contrast

# Function to assign color based on chromosome
def assign_color(chrom_number):
    return color1 if chrom_number % 2 == 0 else color2

# Apply the function to create a color column
sumstat['color'] = sumstat['#CHROM'].apply(assign_color)

chrom_midpoints = []
for chrom, group_df in sumstat.groupby('#CHROM'):
    midpoint = group_df['cumulative_pos'].min() + (group_df['cumulative_pos'].max() - group_df['cumulative_pos'].min()) / 2
    chrom_midpoints.append(midpoint)

# Plotting
g = sns.relplot(
    data=sumstat,
    x='cumulative_pos',
    y='-logp',
    aspect=2,
    hue='color',
    palette=[color1, color2],
    legend=False,
    size=3 
)


# Set x-axis ticks and labels with reversed chromosome order
chrom_labels = np.sort(sumstat['#CHROM'].unique()) # Reversing the order of chromosomes

# Set the title and x-axis, y-axis labels
g.set(title=f"GWAS Manhattan Plot for {phenotypes_labels[disease]} with {scenario_labels[scenario]}")
g.set(xticks=chrom_midpoints, xticklabels=chrom_labels)
g.set_axis_labels("Chromosome", r"$-\log_{10}(P)$")  # Set y-axis label with subscript

# Set the tick labels at a 45-degree angle
for ax in g.axes.flat:
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
line_levels = [5, 6, 7.5, 9]
for ax in g.axes.flat:  # Iterate over all axes
    for level in line_levels:
        ax.axhline(y=level, color='grey', linestyle='--', linewidth=1)

# Optional: Improve the layout
plt.tight_layout()
plt.savefig(f"{path_output}/sumstat_plot_bin_{bin}_{nr_snps}snps.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Define your significance threshold
significance_threshold = -np.log10(5e-8)  # Typically used GWAS significance threshold

# Create a new column for the significance
sumstat['is_significant'] = sumstat['-logp'] > significance_threshold

# Now when plotting, you can use the 'is_significant' column to assign colors
g = sns.relplot(
    data=sumstat,
    x='cumulative_pos',
    y='-logp',
    aspect=2,
    hue='is_significant',  # This will use different colors for significant vs non-significant points
    palette=['#4D4D4D', 'red'],  # Non-significant points in grey, significant in red
    legend=False,
    size=3
)

# ... [Rest of your code for adding horizontal lines, setting labels, and layout] ...

plt.show()

In [ ]:
# Zoom in on chrom 1

In [ ]:
max_pos = 1500
min_pos = 1000

In [ ]:
zoomed = sumstat.sort_values(by='cumulative_pos', ascending=True)
zoomed.reset_index(drop=True, inplace=True)

In [ ]:
zoomed = zoomed.loc[min_pos:max_pos]

In [ ]:
zoomed

In [ ]:

# Define two contrasting colors from the rocket palette
palette = sns.color_palette("rocket", as_cmap=False)
color1 = palette[3]  # First color of the palette
color2 = palette[0] # Last color of the palette, for maximum contrast

# Function to assign color based on chromosome
def assign_color(chrom_number):
    return color1 if chrom_number % 2 == 0 else color2

# Apply the function to create a color column
zoomed['color'] = zoomed['#CHROM'].apply(assign_color)

chrom_midpoints = []
for chrom, group_df in zoomed.groupby('#CHROM'):
    midpoint = group_df['cumulative_pos'].min() + (group_df['cumulative_pos'].max() - group_df['cumulative_pos'].min()) / 2
    chrom_midpoints.append(midpoint)

# Plotting
g = sns.relplot(
    data=zoomed,
    x='cumulative_pos',
    y='-logp',
    aspect=2,
    hue='color',
    palette=[color1, color2],
    legend=False,
    size=3 
)


# Set x-axis ticks and labels with reversed chromosome order
chrom_labels = np.sort(zoomed['#CHROM'].unique()) # Reversing the order of chromosomes

# Set the title and x-axis, y-axis labels
g.set(title=f"GWAS Manhattan Plot for {phenotypes_labels[disease]} with {scenario_labels[scenario]}")
g.set(xticks=chrom_midpoints, xticklabels=chrom_labels)
g.set_axis_labels("Chromosome", r"$-\log_{10}(P)$")  # Set y-axis label with subscript

# Set the tick labels at a 45-degree angle
for ax in g.axes.flat:
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

# Optional: Improve the layout
plt.tight_layout()
plt.savefig(f"{path_output}/sumstat_plot_bin_{bin}_{nr_snps}snps.png", dpi=300, bbox_inches='tight')
plt.show()